# HTML Discrepancy Reports

This notebook is the audit and compliance hand-off. A `DiffResult` is easy to read in a notebook and awkward to hand to anyone else. `veridelta.report` renders it as a single HTML file with no external references: the verdict, the row counts, the column-level drift, and the discrepancy rows themselves, paginated. It opens offline from a CI artifact, an email attachment, or a shared drive.

This notebook builds a comparison with something worth reporting, writes the report, and walks through how to read it. The CLI's `--html` flag produces the same document from YAML; see the [Configuration Guide](../configuration.md#command-line).

## 1. A comparison with something to report

Two billing exports that disagree in every way a report distinguishes: one invoice exists only in the target, one only in the source, and two shared invoices drifted in different columns. A tolerance rule forgives a one-cent rounding difference so the report shows a rule at work, not just raw inequality.

In [ ]:
import polars as pl

from veridelta import DiffConfig, DiffEngine, DiffRule
from veridelta.report import render_html, write_html

# Source: last night's billing export
source = pl.DataFrame(
    {
        "invoice_id": ["INV-001", "INV-002", "INV-003", "INV-004", "INV-005"],
        "amount": [100.00, 250.50, 75.25, 310.00, 42.00],
        "status": ["paid", "open", "paid", "void", "open"],
    }
)

# Target: the replacement pipeline. INV-005 is gone, INV-006 is new, INV-002 moved
# by a cent, INV-003 by a dollar, and INV-004 changed its status casing.
target = pl.DataFrame(
    {
        "invoice_id": ["INV-001", "INV-002", "INV-003", "INV-004", "INV-006"],
        "amount": [100.00, 250.51, 76.25, 310.00, 18.00],
        "status": ["paid", "open", "paid", "VOID", "open"],
    }
)

config = DiffConfig(
    primary_keys=["invoice_id"],
    rules=[DiffRule(column_names=["amount"], absolute_tolerance=0.01)],
)

# The engine consumes LazyFrames, so in-memory DataFrames are wrapped with .lazy()
result = DiffEngine(config, source.lazy(), target.lazy()).run()
print(result.summary.report_summary)

# Output:
# Veridelta Execution Summary
# ===========================
# Status:        FAILED
# Match Rate:    20.0%
# Source Rows:   5
# Target Rows:   5
# Volume Shift:  +0 rows
#
# Row-Level Discrepancies:
# ---------------------------
# Added:         1
# Removed:       1
# Changed:       2
# Total Issues:  4
#
# Top Column-Level Drifts:
# ---------------------------
# - amount: 1 mismatches
# - status: 1 mismatches

## 2. Write the report

`write_html` takes the result and a destination, creates any missing parent directories, and returns the path it wrote. Styles and the small paging script are embedded, so the file is complete on its own.

In [ ]:
report_path = write_html(result, "reports/nightly.html")

print(report_path)
print(f"{report_path.stat().st_size / 1024:.0f} KiB, no external references")

# Output:
# reports/nightly.html
# 6 KiB, no external references

## 3. Reading the report

Open `reports/nightly.html` in a browser. From top to bottom:

* **Verdict badge**: `PASSED` or `FAILED`, the same `is_match` decision the CLI turns into an exit code, judged against `threshold`. The generation time sits beside it.
* **Six cards**: match rate, source and target row counts, and the added, removed, and changed counts. These are the `DiffSummary` fields, so they agree with `report_summary` above.
* **Column-level drift**: one row per compared column that mismatched at least once, ranked by mismatch count. Columns that matched everywhere are left out.
* **Changed rows**, **Added rows**, **Removed rows**: the discrepancy frames, paginated. Added and removed rows are complete records. Changed rows carry both values side by side as `<column>_source` and `<column>_target`, plus one `<column>_is_match` flag per compared column, so a `false` flag names the column that failed for that row.

The changed table is the `result.changed` frame; the flags below are the same ones the report displays.

In [ ]:
flags = result.changed.select("invoice_id", "amount_is_match", "status_is_match")
print(flags.sort("invoice_id"))

# Output:
# shape: (2, 3)
# ┌────────────┬─────────────────┬─────────────────┐
# │ invoice_id ┆ amount_is_match ┆ status_is_match │
# │ ---        ┆ ---             ┆ ---             │
# │ str        ┆ bool            ┆ bool            │
# ╞════════════╪═════════════════╪═════════════════╡
# │ INV-003    ┆ false           ┆ true            │
# │ INV-004    ┆ true            ┆ false           │
# └────────────┴─────────────────┴─────────────────┘

INV-002 is absent: its one-cent drift fell inside the `absolute_tolerance`, so the report counts it as a match. INV-003 failed on `amount` and INV-004 on `status`, which is exactly what the drift table's two single-mismatch rows say.

## 4. Capping large tables

A comparison of ten million rows must not produce a ten-million-row HTML file. Every table is capped at `max_rows` (default 1000) and says so when it has truncated, so a reader never mistakes the visible rows for the whole story. `render_html` returns the document as a string, which is convenient for checking that note here and for attaching a report in CI without touching disk.

In [ ]:
capped = render_html(result, max_rows=1)

note_start = capped.index("Showing the first")
print(capped[note_start : capped.index("</p>", note_start)])

# Output:
# Showing the first 1 of 2 rows. Export artifacts with <code>output_path</code> for the complete set.

Only the changed table was truncated: the added and removed tables held one row each, under the cap. When a report is truncated, set `output_path` on the configuration to export the complete `added`, `removed`, and `changed` frames as files alongside it.

Two notes for pipelines. A report from warehouse pushdown is labeled as primary-keys-only, because the comparison SQL never extracts values; its tables list keys rather than rows. And from the CLI, `veridelta run -c veridelta.yaml --html report.html --html-max-rows 1000` writes this same document without any Python.

---
*(Run the cell below to clean up the report written by this notebook)*

In [ ]:
import shutil

# Remove the report directory (housekeeping)
shutil.rmtree("reports", ignore_errors=True)